# 05. Xem và so sánh kết quả các thí nghiệm E

**Phần A** (không cần GPU): nạp kết quả, bảng so sánh có đánh dấu ★ ở giá trị tốt nhất, thứ hạng từng chỉ số, so sánh theo cặp, giá trị tại các mốc epoch, một hình tổng hợp.
**Phần B** (GPU T4): số tham số / GFLOPs / độ trễ, đánh giá trên tập test, ảnh dự đoán.

**Đầu vào:** `E0.zip`, `E_main.zip`, `E1.zip`, `E2.zip` (và `E3.zip` nếu có) do `ssr_lib.nen_ket_qua` tạo sau mỗi lần chạy.
Trên Kaggle: tải 4 file lên một Dataset rồi Add Input (hoặc Add Input output của notebook 03). Trên máy cá nhân: đặt vào một thư mục và sửa `GOC_KET_QUA` ở cell A1.
Phần B cần Accelerator = GPU T4, Internet = On, Add Input = `seedlingstemrow`.

In [ ]:
import os, sys, subprocess, importlib
GITHUB_USER, TEN_REPO = "Klark352", "ssr-btl-thigiacmay"
if os.path.isdir("/kaggle"):
    DUONG_DAN = f"/kaggle/working/{TEN_REPO}"
    if os.path.isdir(DUONG_DAN):
        subprocess.run(["git", "-C", DUONG_DAN, "pull"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{GITHUB_USER}/{TEN_REPO}.git", DUONG_DAN], check=True)
    SRC = f"{DUONG_DAN}/src"
else:
    SRC = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
sys.path.insert(0, SRC)
import ket_qua as K
importlib.reload(K)

## A1. Nạp kết quả

In [ ]:
GOC_KET_QUA = None        # ví dụ: r"E:\btl_thigiacmay\ket_qua_tai_ve"; None = tự tìm trong /kaggle/input
GHI_RA = "/kaggle/working/ket_qua" if os.path.isdir("/kaggle") else os.path.abspath("ket_qua")
GIAI_NEN = os.path.join(GHI_RA, "_giai_nen")

ung_vien = [GOC_KET_QUA] if GOC_KET_QUA else (["/kaggle/working", "/kaggle/input"] if os.path.isdir("/kaggle") else [os.getcwd()])
ds = {}
for g in ung_vien:
    ds = K.giai_nen_va_tim(g, GIAI_NEN)
    if ds:
        break
if not ds:
    raise SystemExit("Không tìm thấy kết quả. Kiểm tra GOC_KET_QUA hoặc Add Input.")
bang = K.bang_tong_hop(ds)
cap = K.so_sanh_cap(bang)
K.luu(bang, cap, GHI_RA)

## A2. So sánh các E (tập val, lần kiểm định cuối với `best.pt`)

★ là giá trị tốt nhất của cột.

In [ ]:
display(K.bang_danh_dau(bang))
print()
K.xep_hang(bang)

## A3. So sánh theo cặp

Δ dương nghĩa là mô hình đầu cao hơn mô hình đối chứng. Ba cặp ứng với ba câu hỏi ở mục 3.4: toàn bộ thành phần mới (E_main so với E0), riêng DGST (E_main so với E2), tốc độ học (E_main so với E1).

In [ ]:
cot = ["Cặp", "ΔmAP50", "ΔmAP50 (%)", "ΔmIoU", "ΔmIoU (%)", "ΔP", "ΔR"]
display(cap[[c for c in cot if c in cap.columns]])
print()
K.nhan_xet(bang, cap)        # nhận xét tự động; chênh lệch dưới 0,01 được cảnh báo là có thể chỉ là nhiễu

## A4. Giá trị tại các mốc epoch và hình tổng hợp

In [ ]:
display(K.theo_epoch(ds, moc=(1, 5, 10, 15, 20)).round(4))
K.hinh_so_sanh(ds, bang, GHI_RA)

## B. Đánh giá cần GPU (Kaggle)

Mã gọi vào API của nhánh Ultralytics đa nhiệm của tác giả và **chưa được kiểm thử trên repo thật**. Nếu một bước lỗi, gửi nguyên văn thông báo lỗi để chỉnh.

In [ ]:
import setup_env, ssr_lib as L, danh_gia_nang as N
for m in (L, N):
    importlib.reload(m)
setup_env.cai_dat()
try:
    L.chuan_bi_du_lieu()
except RuntimeError as loi:
    print("CẢNH BÁO:", loi)
L.tao_yaml_du_lieu()

### B1. Số tham số, GFLOPs, độ trễ (ảnh 640×640, batch 1)

In [ ]:
phuc_tap = N.do_phuc_tap(ds, imgsz=640, lan=100, luu_vao=GHI_RA)
display(phuc_tap)

### B2. Đánh giá trên tập test

Chạy lại trên val trước để kiểm tra chéo với log (lệch dưới 0,01 thì tin được kết quả test).

In [ ]:
# Chỉ đánh giá test cho 2 mô hình chính; bỏ bước kiểm tra chéo val
chon = {k: ds[k] for k in ("E_main", "E0") if k in ds}
bang_test = N.danh_gia_tat_ca(chon, split="test", luu_vao=GHI_RA)
display(K.bang_danh_dau(bang_test))
K.xep_hang(bang_test)

### B3. Ảnh dự đoán (E_main và E0 trên cùng các ảnh test khó)

In [ ]:
anh = N.chon_anh_kho(so_anh=12)
for ten in ("E_main", "E0"):
    if ten in ds and ds[ten].get("best"):
        thu_muc = N.du_doan(ten, ds[ten]["best"], anh)
        if thu_muc:
            N.ghep_luoi(thu_muc, so_cot=3, luu_vao=f"{GHI_RA}/du_doan_{ten}.png")